 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submisison 
Name           : Ríona Greally                <br>
Student Number : L00196662         <br>
Due Date       : 12th May 2026                <br>
Assignment     : CA2             <br>
Module         : AI for Vision and NLP    <br>
Course         : Postgraduate Diploma in Big Data Analytics and AI

## NLP and Vision Pipeline : High Level
An image of your working pipeline at high level can be inserted here



# Initialisation
Perform pip installs(or use a requirements.txt) <br>
perform imports

## Install packages

In [25]:
# pip installs
pip install pymupdf
pip install opencv-python
pip install matplotlib
pip install numpy panda
pip install nltk
pip install scikit-learn
pip install spacy
python -m spacy download en_core_web_sm

SyntaxError: invalid syntax (2632517952.py, line 2)

## Imports

In [2]:
# imports
import sys
print(sys.executable)
import pymupdf
import os
import site
print(site.getusersitepackages())
import cv2
import matplotlib.pyplot as plt
# allows inline plotting
%matplotlib inline

import pytesseract
from pytesseract import Output
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

import spacy
nlp = spacy.load("en_core_web_sm")


import nltk
nltk.download('punkt_tab')

from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# regular expression
import re

C:\Users\rgreally\AppData\Local\anaconda3\python.exe
C:\Users\rgreally\AppData\Roaming\Python\Python313\site-packages


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\rgreally\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rgreally\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\rgreally\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Support Functions

In [29]:
plt.rcParams['figure.figsize'] = (20, 20)
custom_config = r'--oem 3 --psm 6'




# OCR

In [49]:
#opening PDF for processing
doc = pymupdf.open("setup_tesseract_OCR.pdf")

#saving mulitple pages from the pdf doc as images
for page_num, page in enumerate(doc):
    pix = page.get_pixmap(dpi=200)
    pix.save(f"TextCleansing_page_{page_num + 1}.jpg")

# store all extracted text
all_text = ""

#looping through eaaach image saved from the pdf
for page_num in range(len(doc)):
    img = cv2.imread(f"TextCleansing_page_{page_num + 1}.jpg")
    print(f"Page {page_num + 1} shape:", img.shape)
    
    #Converting image to String
    st = pytesseract.image_to_string(img, config=custom_config)
  
    print(f"\n--- Page {page_num + 1} Text ---\n")
    print(st)  # show first 300 characters of each page 

    #appending extracted text from current page to text from the full doc
    all_text += st


Page 1 shape: (2200, 1700, 3)

--- Page 1 Text ---

et ie il Soha Letterkenny Campu: Killybegs Campu:
Tesseract OCR Set up
Pre-requisites:
e Anaconda Navigator installed and set up
e Jupyter notebook working
e Windows platform
Tesseract OCR Setup Guide (Windows + Conda + Jupyter + requirements.txt)
e Create a Conda environment
e Install all Python OCR libraries from a requirements.txt file
e Install Tesseract OCR (Windows)
e Configure Jupyter Notebook to use Tesseract
e Runa working OCR test
This guide works on restricted laptops.
Note some important clarification about Tesseract:
e Tesseract is an OCR engine — a program that can read text from images
e It is free and open source and maintained by Google
e It is not a Python package
e = It is not a machine-learning library
e It is not installed with pip install pytesseract (this is a python wrapper library)
e It does not depend on NumPy or OpenCV(these libraries are used before OCR to load or
preprocess images)
Tesseract is an Optical 

# NLP

## Tokenisation


In [63]:
#applying nlp to the full extraxted text from document
nlp_allTxt = nlp(all_text)

#looping through fist 20 tokend from the doc and printing them
for token in nlp_allTxt[:20]:
    print(token.text)


et
ie
il
Soha
Letterkenny
Campu
:
Killybegs
Campu
:


Tesseract
OCR
Set
up


Pre
-
requisites
:


In [64]:
#Total number of tokens in the document
len(nlp_allTxt)

935

## Stop Words

In [61]:
#loading english stopwords from NLTK
stop_words = set(stopwords.words('english'))

#converting tokens to lowercase and excluding words that add no meaning 
tokens_without_sw = [token.text for token in nlp_allTxt if token.text.lower() not in stop_words]

#Total of number from document after stopwords removed
len(tokens_without_sw)


758

## Lemmatisation

In [34]:
#Printing first 50 Tokens along with their lemma, POS tag, and description tags
for token in nlp_allTxt[:50]:
    print(f"{token.text:{20}} {token.lemma_:{20}} {token.pos_:{10}} {token.tag_:{5}} {spacy.explain(token.tag_)}")


lemmatised_tokens = [token.lemma_ for token in nlp_allTxt if not token.is_stop and token.is_alpha]


et                   et                   PROPN      NNP   noun, proper singular
ie                   ie                   X          FW    foreign word
il                   il                   PROPN      NNP   noun, proper singular
Soha                 Soha                 PROPN      NNP   noun, proper singular
Letterkenny          Letterkenny          PROPN      NNP   noun, proper singular
Campu                Campu                PROPN      NNP   noun, proper singular
:                    :                    PUNCT      :     punctuation mark, colon or ellipsis
Killybegs            Killybegs            PROPN      NNP   noun, proper singular
Campu                Campu                PROPN      NNP   noun, proper singular
:                    :                    PUNCT      :     punctuation mark, colon or ellipsis

                    
                    SPACE      _SP   whitespace
Tesseract            Tesseract            PROPN      NNP   noun, proper singular
OCR                 

## TF-IDR


In [60]:
#convert cleaned lemmatised tokens back into a single text string to use for TF-IDF
clean_text = " ".join(lemmatised_tokens)

#Create an instance of the TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

#applying TF-IDF to the cleaned text
tfidf_matrix = tfidf_vectorizer.fit_transform([clean_text])

#SAving the important words as features
features = tfidf_vectorizer.get_feature_names_out()

#Convert TF-IDF matrix into scores
tfidf_scores = tfidf_matrix.toarray()[0]

# Create Dataframe to store words and their importance scores
df_tfidf = pd.DataFrame({
    "Word": features,
    "Score": tfidf_scores
})

# Sort words by highest importance
df_sorted = df_tfidf.sort_values(by="Score", ascending=False)

df_sorted.head(10)

,Word,Score
189,tesseract,0.372572
157,python,0.292735
91,kernel,0.292735
89,jupyter,0.239511
129,ocr,0.239511
82,install,0.239511
127,notebook,0.186286
3,add,0.159674
34,conda,0.159674
103,library,0.159674


# Vision

## Sub Heading 1

In [38]:
# code here...

# Multi-modal

## Sub Heading 1

In [39]:
# code here

# Final Output

In [40]:
# code